# 📄 NewsData.io API Documentation
**Topic: Renewable Energy Article Search**
**API Version:** v1
**Last Updated:** 2025-06-01

---

## 🔍 Overview

This API allows clients to search for news articles related to **renewable energy** using the [NewsData.io](https://newsdata.io/) service. It supports various query parameters such as keywords, language, country, category, and publication date.

---

## 🛠️ Base URL

```
https://newsdata.io/api/1/news
```

---

## 🔐 Authentication

All requests must include a valid API key as a query parameter:

```
?apikey=YOUR_API_KEY
```

---

## 📥 Request Format

### Example Request (Search "renewable energy" articles in English):

```http
GET /api/1/news?apikey=YOUR_API_KEY&q=renewable+energy&language=en&category=environment
Host: newsdata.io
```

### Curl Example

```bash
curl "https://newsdata.io/api/1/news?apikey=YOUR_API_KEY&q=renewable+energy&language=en&category=environment"
```

---

## 🔧 Query Parameters

| Parameter     | Type     | Description                                                                 |
|---------------|----------|-----------------------------------------------------------------------------|
| `apikey`      | string   | **Required.** Your NewsData.io API key.                                     |
| `q`           | string   | Search keywords (e.g., "renewable energy").                                 |
| `language`    | string   | Filter articles by language (e.g., `en` for English).                       |
| `country`     | string   | Filter by country code (e.g., `us`, `gb`, `de`).                            |
| `category`    | string   | Filter by category (e.g., `environment`, `technology`).                     |
| `from_date`   | string   | Filter articles from this date (format: `YYYY-MM-DD`).                      |
| `to_date`     | string   | Filter articles up to this date (format: `YYYY-MM-DD`).                     |
| `page`        | integer  | Pagination - get specific page of results.                                  |

---

## 📤 Response Format

### JSON Response Example

```json
{
  "status": "success",
  "totalResults": 120,
  "results": [
    {
      "title": "How Solar Energy is Powering Rural Communities",
      "link": "https://example.com/article",
      "pubDate": "2025-05-30 10:00:00",
      "source_id": "renewabletimes",
      "category": ["environment", "energy"],
      "language": "en",
      "country": "us",
      "creator": ["Jane Doe"]
    }
  ]
}
```

---

## 📛 Error Handling

| Status Code | Message                   | Description                                 |
|-------------|---------------------------|---------------------------------------------|
| 401         | `Invalid API key`         | The provided API key is incorrect.          |
| 429         | `Rate limit exceeded`     | Too many requests; wait and try again.      |
| 500         | `Internal server error`   | Something went wrong on NewsData.io’s side. |

---

## 🧠 Tips for Renewable Energy Search

- Use keywords like `"solar energy"`, `"wind power"`, `"clean energy"`, `"renewable projects"`.
- Combine with `category=environment` for best results.
- Use date filters to get the latest trends (e.g., `from_date=2025-05-01`).

---

## 🗂️ Sample Use Case

**Fetch all English articles about solar panels published in May 2025:**

```http
GET /api/1/news?apikey=YOUR_API_KEY&q=solar+panels&language=en&from_date=2025-05-01&to_date=2025-05-31
```

---



In [ ]:
!pip install newsdataapi

In [5]:
import os
import http.client
import urllib.parse
import json
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

NEWS_DATAIO_API_KEY = os.getenv("NEWS_DATAIO_API_KEY")

# Define connection
conn = http.client.HTTPSConnection('newsdata.io')

# Build query parameters
params = urllib.parse.urlencode({
    'apikey': NEWS_DATAIO_API_KEY,
    'q': 'renewable energy'
})

# Make GET request
conn.request('GET', f'/api/1/latest?{params}')

# Get response
res = conn.getresponse()
data = res.read()

# Decode JSON response
json_data = json.loads(data.decode('utf-8'))

# Extract articles and build the source-to-url mapping
source_url_map = {
    article.get('article_id', 'unknown'): article.get('link')
    for article in json_data.get('results', [])
}

# Print the result
print(json.dumps(source_url_map, indent=2))


{
  "65b6e45fa622ac38bcde300f7ee3dfdc": "https://menafn.com/1109620534/LG-Energy-Begins-Mass-Production-Of-Batteries-At-US-Plant",
  "20194db65c7b36b700f0ae7283d74f19": "https://nypost.com/2025/05/31/opinion/michael-goodwin-jake-tapper-splits-from-dem-allies-in-latest-book-detailing-bidens-cognitive-decline/",
  "ddd55af8a97690e8c7c27d1e88e37ae7": "https://www.vanguardngr.com/2025/06/nnaji-highlights-tinubus-strides-in-energy-innovation-and-jobs-in-two-years/",
  "a59f282e0d1373cfeee2a9b8184186e1": "https://mb.com.ph/article/10869464/world/trump-fast-tracks-utah-uranium-mine-but-industry-revival-may-wait-for-higher-prices",
  "acfe8935ff4c07e81016901d67e176ff": "https://www.thestar.com.my/news/focus/2025/06/01/between-the-eagle-and-the-dragon",
  "f6e89386499e33aaeccbcf1d3657b4e1": "https://www.straitstimes.com/asia/east-asia/koreas-presidential-front-runner-lee-jae-myung-backs-nuclear-power-for-now",
  "21dd4114e33e78547f58d2d9d22f7455": "http://island.lk/rwanda-at-a-glance-lessons-fo

### 📰 Using [newspaper](https://pypi.org/project/newspaper/) lib and selenium

In [6]:
import json
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from newspaper import Article

def scrape_with_newspaper_selenium(source_url_map):
    options = Options()
    options.add_argument("--headless")
    options.add_argument("user-agent=Mozilla/5.0")

    results = []

    for source, url in source_url_map.items():
        try:
            driver = webdriver.Chrome(options=options)
            driver.get(url)
            time.sleep(3)

            html = driver.page_source
            driver.quit()

            article = Article(url)
            article.set_html(html)
            article.download_state = 2  # Force state to SUCCESS
            article.parse()

            result = {
                "source": source,
                "title": article.title,
                "url": url,
                "authors": article.authors,
                "published_date": article.publish_date.isoformat() if article.publish_date else None,
                "paragraphs": [p.strip() for p in article.text.split("\n") if p.strip()]
            }

            results.append(result)

        except Exception as e:
            print(f"Error scraping {url} from {source}: {e}")

    print(json.dumps(results, indent=2))

In [7]:
scrape_with_newspaper_selenium(source_url_map)


[
  {
    "source": "65b6e45fa622ac38bcde300f7ee3dfdc",
    "title": "LG Energy Begins Mass Production Of Batteries At US Plant",
    "url": "https://menafn.com/1109620534/LG-Energy-Begins-Mass-Production-Of-Batteries-At-US-Plant",
    "authors": [],
    "published_date": null,
    "paragraphs": [
      "MENAFN - IANS) Seoul, June 1 (IANS) LG Energy Solution Ltd (LGES), South Korea's leading battery maker, said on Sunday it has begun mass production of lithium iron phosphate (LFP) batteries for energy storage systems (ESS) at its manufacturing plant in the United States.",
      "The pouch-type LFP batteries for ESS, based on long cell technology, are being manufactured at the LGES plant in Michigan, according to the Korean company, reports Yonhap news agency.",
      "\"We are currently in discussions with multiple customers in the North American region for the supply of our ESS batteries,\" LGES officials said, noting that supply to a number of major U.S. energy firms, such as Terra-

### 📰 Using news beautiful soap and selenium

In [10]:
from bs4 import BeautifulSoup

# Custom headers and cookies
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3',
}

COOKIES = {
    'name_of_the_consent_cookie': 'value_indicating_consent',
}

def scrape_articles_with_selenium_bs4(source_url_map):
    results = []

    for source, url in source_url_map.items():
        try:
            options = Options()
            options.add_argument("--headless")
            options.add_argument(f"user-agent={HEADERS['User-Agent']}")

            driver = webdriver.Chrome(options=options)
            driver.get(url)

            # Inject cookies if needed
            for name, value in COOKIES.items():
                driver.add_cookie({'name': name, 'value': value, 'domain': '.' + url.split('/')[2]})

            driver.get(url)  # Reload page with cookies
            time.sleep(3)

            html = driver.page_source
            driver.quit()

            soup = BeautifulSoup(html, "html.parser")

            # Extract title
            title_tag = soup.find("h1") or soup.title
            title = title_tag.get_text(strip=True) if title_tag else "N/A"

            # Extract paragraphs
            paragraphs = [p.get_text(strip=True) for p in soup.find_all("p") if p.get_text(strip=True)]

            # Extract author
            author_tag = soup.find("meta", {"name": "author"})
            author = author_tag["content"] if author_tag else "Unknown"

            # Extract published date
            published_date = None
            date_meta = soup.find("meta", {"property": "article:published_time"}) or \
                        soup.find("meta", {"name": "date"}) or \
                        soup.find("meta", {"itemprop": "datePublished"}) or \
                        soup.find("time")

            if date_meta:
                if date_meta.has_attr("content"):
                    published_date = date_meta["content"]
                elif date_meta.has_attr("datetime"):
                    published_date = date_meta["datetime"]
                else:
                    published_date = date_meta.get_text(strip=True)

            result = {
                "source": source,
                "title": title,
                "url": url,
                "author": author,
                "published_date": published_date,
                "paragraphs": paragraphs
            }

            results.append(result)

        except Exception as e:
            print(f"Error scraping {url} ({source}): {e}")

    print(json.dumps(results, indent=2))


In [11]:
scrape_articles_with_selenium_bs4(source_url_map)


Error scraping https://mb.com.ph/article/10869464/world/trump-fast-tracks-utah-uranium-mine-but-industry-revival-may-wait-for-higher-prices (a59f282e0d1373cfeee2a9b8184186e1): Message: invalid cookie domain
  (Session info: chrome=136.0.7103.116)
Stacktrace:
0   chromedriver                        0x000000010d190898 chromedriver + 5986456
1   chromedriver                        0x000000010d1879ca chromedriver + 5949898
2   chromedriver                        0x000000010cc40443 chromedriver + 414787
3   chromedriver                        0x000000010ccf1704 chromedriver + 1140484
4   chromedriver                        0x000000010ccb8312 chromedriver + 906002
5   chromedriver                        0x000000010ccdf566 chromedriver + 1066342
6   chromedriver                        0x000000010ccb80e3 chromedriver + 905443
7   chromedriver                        0x000000010cc8461d chromedriver + 693789
8   chromedriver                        0x000000010cc85281 chromedriver + 696961
9   chro

### 📡 Using requests and trafitula

In [14]:
import json
import requests
import trafilatura

def scrape_multiple_with_trafilatura(source_url_map):
    results = []

    for source, url in source_url_map.items():
        try:
            # Fetch the raw HTML from the URL
            response = requests.get(url, timeout=10)
            response.raise_for_status()
            html = response.text
        except Exception as e:
            print(f"Failed to fetch the page from {url} ({source}): {e}")
            continue

        # Extract structured content with metadata
        downloaded = trafilatura.extract(
            html,
            include_comments=False,
            include_tables=False,
            output_format='json'
        )

        if downloaded is None:
            print(f"Failed to extract content from {url} ({source})")
            continue

        data = json.loads(downloaded)

        # Extract clean paragraphs
        paragraphs = data.get('text', '').split('\n') if 'text' in data else []

        result = {
            "source": source,
            "title": data.get("title", "Unknown"),
            "url": url,
            "authors": data.get("author", []),
            "published_date": data.get("date", "Unknown"),
            "paragraphs": [p.strip() for p in paragraphs if p.strip()]
        }

        results.append(result)

    print(json.dumps(results, indent=2, ensure_ascii=False))

In [15]:
scrape_multiple_with_trafilatura(source_url_map)

Failed to fetch the page from https://www.vanguardngr.com/2025/06/nnaji-highlights-tinubus-strides-in-energy-innovation-and-jobs-in-two-years/ (ddd55af8a97690e8c7c27d1e88e37ae7): 403 Client Error: Forbidden for url: https://www.vanguardngr.com/2025/06/nnaji-highlights-tinubus-strides-in-energy-innovation-and-jobs-in-two-years/
Failed to fetch the page from https://mb.com.ph/article/10869464/world/trump-fast-tracks-utah-uranium-mine-but-industry-revival-may-wait-for-higher-prices (a59f282e0d1373cfeee2a9b8184186e1): 403 Client Error: Forbidden for url: https://mb.com.ph/article/10869464/world/trump-fast-tracks-utah-uranium-mine-but-industry-revival-may-wait-for-higher-prices
Failed to fetch the page from https://financialpost.com/pmn/business-pmn/koreas-presidential-front-runner-backs-nuclear-power-for-now (1198454895b89df1a09f2f6287b6f199): 403 Client Error: Forbidden for url: https://financialpost.com/pmn/business-pmn/koreas-presidential-front-runner-backs-nuclear-power-for-now
[
  {
 